# Legacy MCS recalculation (seed-aligned, `arch.bootstrap.MCS`)

**Specification:** `MCS_LEGACY_RESULTS_RECALCULATION_DESIGN.md` (branch `claude/mcs-legacy-results-v1`).

## Scope and locked decisions

This notebook **consumes the legacy per-seed results and recalculates the Model Confidence Set (MCS) only.**

1. The legacy result set (`article_results_3p/per_seed_metrics_3p.csv`) stays the primary source for the thesis. It is read only.
2. The divergent run `SBTS / asian_worst_of_put / κ=0.95 / canonical seed=3` is represented by the converged retry at **actual seed 10**. That replacement stays in the descriptive summaries (ten replicates).
3. For MCS every row of the `T × k` loss matrix must be one common observational unit. Rows are therefore keyed by **actual seed**, and each cell uses the intersection of actual seeds across `GBM`, `Heston`, `SBTS`:
   * `asian_worst_of_put`, κ = 0.95 (3 periods): common actual seeds `0,1,2,4,5,6,7,8,9` → **n = 9**
   * all other 15 cells: `0,…,9` → **n = 10**
4. MCS: phase `cvar`, loss `std` (σ(R)), α = 0.10, `arch.bootstrap.MCS`, primary method `R`, sensitivity `max`, circular bootstrap, block size 1, 50,000 replications, SHA-256-derived cell seeds.
5. The hand-written `mcs_seed_level` routine from `article_4_3period.ipynb` is **superseded and not used**.

**Not done here:** no training, no checkpoint loading, no path generation, no OOS evaluation, no GPU, no market-data download, no change to descriptive tables, and **no recalculation of the 216 paired t-tests**. A corrected MCS does not certify that the paired t-tests are correctly aligned.

All outputs go to `article_results_3p/mcs_recalculated_v2/`; legacy files are never written.

**Run:** *Runtime → Run all* in Colab. Runtime is ≈ 1–2 min on CPU. The last cell prints `FINAL GATE: PASS` or raises.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 2 — Frozen estimand constants (pure; no I/O)
#
# Every value below is locked by MCS_LEGACY_RESULTS_RECALCULATION_DESIGN.md
# §3–§6. Do not edit them to make a result "look right".
# ═══════════════════════════════════════════════════════════════════
MODEL_ORDER = ("GBM", "Heston", "SBTS")
PERIODS = ("COVID_2019_2020", "PostCOVID_2021_22", "Recent_2023_25")
OPTIONS = ("basket_asian_call", "asian_worst_of_put")
KAPPAS = (0.95, 1.00, 1.05)
CANONICAL_SEEDS = tuple(range(10))

REQUIRED_COLUMNS = ("ds", "option", "kappa", "seed", "phase", "period",
                    "std", "cvar95", "mean", "V0")

# The single approved replacement: (ds, option, kappa, canonical_seed) -> actual_seed
SEED_REPLACEMENTS = {
    ("SBTS", "asian_worst_of_put", 0.95, 3): 10,
}

# MCS estimand (§4)
PHASE_RUN = "cvar"
LOSS_COL = "std"
ALPHA = 0.10
REPS = 50_000
BLOCK_SIZE = 1
BOOTSTRAP = "circular"
PRIMARY_METHOD = "R"
SENSITIVITY_METHOD = "max"
RNG_KEY_VERSION = "mcs-v2"

# Expected alignment (§5): affected cells use 9 common actual seeds, others 10
AFFECTED_COMMON_SEEDS = (0, 1, 2, 4, 5, 6, 7, 8, 9)
FULL_COMMON_SEEDS = tuple(range(10))
AFFECTED_CELLS = tuple((p, "asian_worst_of_put", 0.95) for p in PERIODS)

OUTPUT_FILES = (
    "mcs_corrected_method_R.csv",
    "mcs_sensitivity_method_max.csv",
    "mcs_pvalues_long.csv",
    "mcs_alignment_audit.csv",
    "mcs_comparison_with_legacy.csv",
    "mcs_metadata.json",
    "MCS_RECALCULATION_REPORT.md",
    "tab_mcs_corrected.tex",
)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 3 — Environment: imports, package versions, paths
#
# Paths are configurable here. Optional environment overrides (for local
# runs): MCS_ARTICLE_ROOT, MCS_INPUT_CSV, MCS_LEGACY_MCS_CSV.
# ═══════════════════════════════════════════════════════════════════
import datetime
import json
import os
import platform
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

try:
    import arch
except ImportError:
    if not IN_COLAB:
        raise ImportError("Package 'arch' is required: pip install -r requirements.txt")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "arch>=7.0"])
    import arch

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

# ── Paths ────────────────────────────────────────────────────────
ARTICLE_ROOT = Path(os.environ.get("MCS_ARTICLE_ROOT", "/content/drive/MyDrive/ARTICLE_SBTS"))
INPUT_CSV = Path(os.environ.get("MCS_INPUT_CSV",
                                ARTICLE_ROOT / "article_results_3p" / "per_seed_metrics_3p.csv"))
LEGACY_RESULTS_DIR = INPUT_CSV.parent                     # read-only
LEGACY_MCS_CSV = Path(os.environ.get("MCS_LEGACY_MCS_CSV",  # optional, comparison only
                                     LEGACY_RESULTS_DIR / "mcs_3p.csv"))
OUTPUT_DIR = LEGACY_RESULTS_DIR / "mcs_recalculated_v2"   # the only place we write

VERSIONS = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "arch": arch.__version__,
}


def _git_commit():
    if os.environ.get("MCS_GIT_COMMIT"):
        return os.environ["MCS_GIT_COMMIT"]
    try:
        return subprocess.check_output(["git", "rev-parse", "HEAD"], text=True,
                                       stderr=subprocess.DEVNULL).strip()
    except Exception:
        return None


GIT_COMMIT = _git_commit()

print(f"In Colab        : {IN_COLAB}")
print(f"Versions        : {VERSIONS}")
print(f"Git commit      : {GIT_COMMIT}")
print(f"Input CSV       : {INPUT_CSV}")
print(f"Legacy MCS CSV  : {LEGACY_MCS_CSV}  (comparison only; never an input)")
print(f"Output dir      : {OUTPUT_DIR}")
print(f"\nMCS: phase={PHASE_RUN}, loss={LOSS_COL}, alpha={ALPHA}, reps={REPS:,}, "
      f"block_size={BLOCK_SIZE}, bootstrap={BOOTSTRAP}, "
      f"primary={PRIMARY_METHOD}, sensitivity={SENSITIVITY_METHOD}")
print(f"Model order     : {MODEL_ORDER}")
print(f"Seed replacement: {SEED_REPLACEMENTS}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 4 — Pure helper functions (no Drive access, no global state)
#
# Everything the execution cells need lives here so that the same code
# is exercised by the synthetic smoke tests (CELL 9) and by
# tests/test_mcs_recalculation.py.
# ═══════════════════════════════════════════════════════════════════
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from arch.bootstrap import MCS


def sha256_file(path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def snapshot_tree(root, exclude=None) -> dict:
    """SHA-256 of every file under `root` (relative path -> hash), skipping `exclude`."""
    root = Path(root)
    exclude = Path(exclude).resolve() if exclude is not None else None
    out = {}
    if not root.is_dir():
        return out
    for f in sorted(root.rglob("*")):
        if not f.is_file():
            continue
        if exclude is not None and (f.resolve() == exclude or exclude in f.resolve().parents):
            continue
        out[str(f.relative_to(root))] = sha256_file(f)
    return out


def _as_int_series(s: pd.Series, name: str) -> pd.Series:
    num = pd.to_numeric(s, errors="coerce")
    arr = num.to_numpy(dtype=float)
    if np.isnan(arr).any() or not np.isfinite(arr).all() or not (arr == np.round(arr)).all():
        raise ValueError(f"Column {name!r} must contain finite integer values only")
    return num.astype("int64")


def _seed_list(values) -> str:
    return ",".join(str(int(v)) for v in values)


def _kappa_mask(series: pd.Series, kappa: float) -> pd.Series:
    return np.isclose(series.astype(float), float(kappa), rtol=0.0, atol=1e-9)


def cell_rng_seed(period: str, option: str, kappa: float, method: str) -> int:
    """Process-independent bootstrap seed for one (cell, method)."""
    key = f"{RNG_KEY_VERSION}|{period}|{option}|{kappa:.2f}|{method}"
    digest = hashlib.sha256(key.encode("utf-8")).digest()
    return int.from_bytes(digest[:8], "big") % (2**32)


# ── Loading and seed provenance ──────────────────────────────────
def load_legacy_metrics(path: Path) -> tuple[pd.DataFrame, dict]:
    """Read the legacy per-seed CSV. Fails loudly; there is no fallback."""
    path = Path(path).expanduser()
    if not path.is_file():
        raise FileNotFoundError(
            f"Legacy per-seed CSV not found: {path}\n"
            "No fallback is permitted (no canonical_runs/, no new evaluations, "
            "no previous MCS CSV, no glob/mtime selection). Fix the path and re-run.")
    resolved = path.resolve()
    digest = sha256_file(resolved)
    df = pd.read_csv(resolved)
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Legacy CSV is missing required columns: {missing}")
    df = df.copy()
    for c in ("ds", "option", "phase", "period"):
        df[c] = df[c].astype(str)
    df["kappa"] = pd.to_numeric(df["kappa"], errors="raise").astype(float).round(2)
    df["seed"] = _as_int_series(df["seed"], "seed")
    info = {
        "path": str(resolved),
        "sha256": digest,
        "n_rows": int(df.shape[0]),
        "n_cols": int(df.shape[1]),
        "columns": list(df.columns),
    }
    return df, info


def attach_actual_seed(
    df: pd.DataFrame,
    replacements: dict[tuple[str, str, float, int], int],
) -> pd.DataFrame:
    """Add canonical_seed / actual_seed / replacement_used without touching `seed`.

    If `file_seed` exists it is used as actual_seed, but only after checking
    that it agrees with the approved replacement mapping on every row.
    """
    if "canonical_seed" in df.columns or "actual_seed" in df.columns:
        raise ValueError("canonical_seed/actual_seed already present; refusing to overwrite")
    out = df.copy()
    canonical = _as_int_series(out["seed"], "seed")
    expected_actual = canonical.copy()
    for key, actual in replacements.items():
        ds, option, kappa, cseed = key
        mask = ((out["ds"] == ds) & (out["option"] == option)
                & _kappa_mask(out["kappa"], kappa) & (canonical == int(cseed)))
        if not mask.any():
            raise ValueError(f"Approved replacement {key} -> {actual} matches no rows")
        expected_actual[mask] = int(actual)

    if "file_seed" in out.columns:
        file_seed = _as_int_series(out["file_seed"], "file_seed")
        bad = file_seed != expected_actual
        if bad.any():
            cols = ["ds", "option", "kappa", "phase", "period", "seed", "file_seed"]
            raise ValueError(
                "file_seed disagrees with the approved seed mapping on "
                f"{int(bad.sum())} row(s):\n{out.loc[bad, cols].head(10).to_string()}")
        actual = file_seed
    else:
        actual = expected_actual

    out["canonical_seed"] = canonical.astype("int64")
    out["actual_seed"] = actual.astype("int64")
    out["replacement_used"] = out["canonical_seed"] != out["actual_seed"]
    return out


def expected_actual_seeds(ds: str, option: str, kappa: float, replacements) -> list[int]:
    seeds = set(CANONICAL_SEEDS)
    for (r_ds, r_opt, r_kappa, cseed), aseed in replacements.items():
        if r_ds == ds and r_opt == option and abs(r_kappa - kappa) < 1e-9:
            seeds.discard(int(cseed))
            seeds.add(int(aseed))
    return sorted(seeds)


def validate_legacy_metrics(df: pd.DataFrame, replacements) -> pd.DataFrame:
    """Hard validation (§5 steps 1–2 plus coverage). Returns one row per check."""
    checks = []

    def add(name, ok, detail=""):
        checks.append({"check": name, "passed": bool(ok), "detail": str(detail)})

    run = df[df["phase"] == PHASE_RUN]
    add("phase 'cvar' present", len(run) > 0, f"{len(run)} rows")

    unknown = {
        "ds": sorted(set(run["ds"]) - set(MODEL_ORDER)),
        "period": sorted(set(run["period"]) - set(PERIODS)),
        "option": sorted(set(run["option"]) - set(OPTIONS)),
        "kappa": sorted(set(run["kappa"].round(2)) - set(KAPPAS)),
    }
    unknown = {k: v for k, v in unknown.items() if v}
    add("no unknown ds/period/option/kappa in cvar rows", not unknown, unknown or "ok")

    loss = pd.to_numeric(run[LOSS_COL], errors="coerce").to_numpy(dtype=float)
    n_bad = int((~np.isfinite(loss)).sum())
    add(f"all cvar '{LOSS_COL}' values finite", n_bad == 0, f"{n_bad} non-finite")

    key = ["ds", "option", "kappa", "phase", "period"]
    for seed_col in ("canonical_seed", "actual_seed"):
        dup = df.duplicated(key + [seed_col], keep=False)
        add(f"no duplicate (ds, option, kappa, phase, period, {seed_col})",
            not dup.any(), f"{int(dup.sum())} duplicated rows")

    missing_cells, bad_sets = [], []
    for period in PERIODS:
        for option in OPTIONS:
            for kappa in KAPPAS:
                cell = run[(run["period"] == period) & (run["option"] == option)
                           & _kappa_mask(run["kappa"], kappa)]
                for ds in MODEL_ORDER:
                    sub = cell[cell["ds"] == ds]
                    if sub.empty:
                        missing_cells.append((period, option, kappa, ds))
                        continue
                    canon = sorted(sub["canonical_seed"].unique().tolist())
                    act = sorted(sub["actual_seed"].unique().tolist())
                    exp_act = expected_actual_seeds(ds, option, kappa, replacements)
                    if canon != list(CANONICAL_SEEDS) or act != exp_act:
                        bad_sets.append((period, option, kappa, ds, canon, act))
    add("18 cells x 3 models covered", not missing_cells,
        f"{len(missing_cells)} missing: {missing_cells[:5]}")
    add("canonical seeds 0..9 and expected actual seeds in every cell/model",
        not bad_sets, f"{len(bad_sets)} mismatches: {bad_sets[:3]}")
    return pd.DataFrame(checks)


# ── Loss matrix (§5) ─────────────────────────────────────────────
def build_mcs_loss_matrix(
    df: pd.DataFrame,
    period: str,
    option: str,
    kappa: float,
    models=MODEL_ORDER,
    phase: str = PHASE_RUN,
    loss: str = LOSS_COL,
) -> tuple[pd.DataFrame, dict]:
    """T x k loss matrix on the intersection of *actual* seeds.

    Rows are indexed by actual_seed, so one row never mixes different
    replicates (e.g. GBM/Heston actual seed 3 with SBTS actual seed 10).
    """
    models = list(models)
    cell = df[(df["phase"] == phase) & (df["period"] == period)
              & (df["option"] == option) & _kappa_mask(df["kappa"], kappa)]
    per_model, seed_sets = {}, {}
    for m in models:
        sub = cell[cell["ds"] == m]
        if sub.empty:
            raise ValueError(f"No rows for model {m} in cell {(period, option, kappa)}")
        dup = sub["actual_seed"][sub["actual_seed"].duplicated()].unique().tolist()
        if dup:
            raise ValueError(f"Duplicate actual_seed {dup} for {m} in cell {(period, option, kappa)}")
        vals = pd.to_numeric(sub[loss], errors="coerce").to_numpy(dtype=float)
        if not np.isfinite(vals).all():
            raise ValueError(f"Non-finite '{loss}' for {m} in cell {(period, option, kappa)}")
        seed_sets[m] = set(int(s) for s in sub["actual_seed"])
        per_model[m] = sub

    common = sorted(set.intersection(*seed_sets.values()))
    if len(common) < 2:
        raise ValueError(f"Fewer than 2 common actual seeds in cell {(period, option, kappa)}")

    losses = pd.DataFrame(index=pd.Index(common, name="actual_seed"), columns=models, dtype=float)
    audit_models = {}
    for m in models:
        sub = per_model[m].set_index("actual_seed")
        losses[m] = sub.loc[common, loss].astype(float).to_numpy()
        excluded = sorted(seed_sets[m] - set(common))
        reasons = []
        for s in excluded:
            absent = [o for o in models if o != m and s not in seed_sets[o]]
            reasons.append(f"actual seed {s} not observed for {'/'.join(absent)}")
        audit_models[m] = {
            "canonical_seeds": sorted(int(s) for s in per_model[m]["canonical_seed"]),
            "actual_seeds": sorted(seed_sets[m]),
            "excluded_actual_seeds": excluded,
            "exclusion_reason": "; ".join(reasons),
            "n_available": int(len(seed_sets[m])),
            "replacement_count": int(per_model[m]["replacement_used"].sum()),
            "mean_all_available": float(per_model[m][loss].astype(float).mean()),
            "mean_common": float(losses[m].mean()),
        }

    if list(losses.columns) != models:
        raise AssertionError(f"Column order {list(losses.columns)} != {models}")
    arr = losses.to_numpy(dtype=float)
    if arr.shape != (len(common), len(models)) or not np.isfinite(arr).all():
        raise AssertionError("Loss matrix has missing or non-finite entries")
    if losses.index.has_duplicates:
        raise AssertionError("Loss matrix has duplicated actual seeds")

    audit = {
        "period": period, "option": option, "kappa": float(kappa),
        "phase": phase, "loss": loss, "models": models,
        "common_actual_seeds": common, "n_common": len(common),
        "per_model": audit_models,
    }
    return losses, audit


def verify_matrix_pairing(df, losses, period, option, kappa,
                          phase=PHASE_RUN, loss=LOSS_COL) -> bool:
    """Re-derive every matrix entry from the source rows by (ds, actual_seed)."""
    cell = df[(df["phase"] == phase) & (df["period"] == period)
              & (df["option"] == option) & _kappa_mask(df["kappa"], kappa)]
    for s in losses.index:
        for m in losses.columns:
            src = cell[(cell["ds"] == m) & (cell["actual_seed"] == s)]
            if len(src) != 1:
                raise AssertionError(f"{(period, option, kappa)} row {s}/{m}: {len(src)} source rows")
            if float(src[loss].iloc[0]) != float(losses.at[s, m]):
                raise AssertionError(f"{(period, option, kappa)} row {s}/{m}: value mismatch")
    return True


# ── MCS (§4, §7) ─────────────────────────────────────────────────
def run_arch_mcs(
    losses: pd.DataFrame,
    method: str,
    alpha: float,
    reps: int,
    seed: int,
    block_size: int = BLOCK_SIZE,
    bootstrap: str = BOOTSTRAP,
) -> dict:
    """Run arch.bootstrap.MCS and read membership/p-values from the computed object."""
    models = list(losses.columns)
    arr = losses.to_numpy(dtype=float)
    if not np.isfinite(arr).all():
        raise ValueError("Non-finite losses supplied to MCS")
    if losses.index.has_duplicates:
        raise ValueError("Duplicated rows supplied to MCS")
    mcs = MCS(losses, size=alpha, reps=reps, block_size=block_size,
              method=method, bootstrap=bootstrap, seed=seed)
    mcs.compute()
    included_set = set(mcs.included)
    excluded_set = set(mcs.excluded)
    if included_set & excluded_set or (included_set | excluded_set) != set(models):
        raise AssertionError(f"MCS included/excluded do not partition {models}")
    pv = mcs.pvalues["Pvalue"]
    return {
        "method": method, "alpha": float(alpha), "reps": int(reps),
        "block_size": int(block_size), "bootstrap": bootstrap, "rng_seed": int(seed),
        "included_models": [m for m in models if m in included_set],
        "excluded_models": [m for m in models if m in excluded_set],
        "elimination_order": [str(m) for m in pv.index],
        "mcs_size": len(included_set),
        "pvalues": {m: float(pv.loc[m]) for m in models},
    }


def run_mcs_all_cells(df, method, audits_out=None, input_sha256="", arch_version="",
                      alpha=ALPHA, reps=REPS) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Run one MCS method on all 18 cells. Returns (results table, p-values long)."""
    rows, prow = [], []
    for period in PERIODS:
        for option in OPTIONS:
            for kappa in KAPPAS:
                losses, audit = build_mcs_loss_matrix(df, period, option, kappa)
                seed = cell_rng_seed(period, option, kappa, method)
                res = run_arch_mcs(losses, method, alpha, reps, seed)
                if audits_out is not None:
                    audits_out[(period, option, kappa)] = (losses, audit)
                pm = audit["per_model"]
                rows.append({
                    "period": period, "option": option, "kappa": kappa,
                    "loss": LOSS_COL, "phase": PHASE_RUN, "alpha": alpha,
                    "method": method, "bootstrap": BOOTSTRAP,
                    "block_size": BLOCK_SIZE, "reps": reps, "rng_seed": seed,
                    "n_common": audit["n_common"],
                    "common_actual_seeds": _seed_list(audit["common_actual_seeds"]),
                    "included_models": ",".join(res["included_models"]),
                    "excluded_models": ",".join(res["excluded_models"]),
                    "mcs_size": res["mcs_size"],
                    "gbm_mean_common": pm["GBM"]["mean_common"],
                    "heston_mean_common": pm["Heston"]["mean_common"],
                    "sbts_mean_common": pm["SBTS"]["mean_common"],
                    "gbm_mean_all_available": pm["GBM"]["mean_all_available"],
                    "heston_mean_all_available": pm["Heston"]["mean_all_available"],
                    "sbts_mean_all_available": pm["SBTS"]["mean_all_available"],
                    "gbm_pvalue": res["pvalues"]["GBM"],
                    "heston_pvalue": res["pvalues"]["Heston"],
                    "sbts_pvalue": res["pvalues"]["SBTS"],
                    "elimination_order": ",".join(res["elimination_order"]),
                    "affected_cell": (period, option, kappa) in AFFECTED_CELLS,
                    "input_sha256": input_sha256,
                    "arch_version": arch_version,
                })
                for rank, m in enumerate(res["elimination_order"], start=1):
                    prow.append({
                        "period": period, "option": option, "kappa": kappa,
                        "method": method, "rng_seed": seed, "model": m,
                        "elimination_rank": rank, "mcs_pvalue": res["pvalues"][m],
                        "in_mcs": m in res["included_models"],
                        "n_common": audit["n_common"],
                    })
    return pd.DataFrame(rows), pd.DataFrame(prow)


def alignment_audit_table(audits: dict) -> pd.DataFrame:
    rows = []
    for (period, option, kappa), (_, audit) in audits.items():
        exp = AFFECTED_COMMON_SEEDS if (period, option, kappa) in AFFECTED_CELLS else FULL_COMMON_SEEDS
        status = "ok" if tuple(audit["common_actual_seeds"]) == tuple(exp) else "unexpected"
        for ds in audit["models"]:
            pm = audit["per_model"][ds]
            rows.append({
                "period": period, "option": option, "kappa": kappa, "ds": ds,
                "canonical_seeds": _seed_list(pm["canonical_seeds"]),
                "actual_seeds": _seed_list(pm["actual_seeds"]),
                "common_actual_seeds": _seed_list(audit["common_actual_seeds"]),
                "excluded_actual_seeds": _seed_list(pm["excluded_actual_seeds"]),
                "exclusion_reason": pm["exclusion_reason"],
                "n_available": pm["n_available"],
                "n_common": audit["n_common"],
                "replacement_count": pm["replacement_count"],
                "expected_common_actual_seeds": _seed_list(exp),
                "alignment_status": status,
            })
    return pd.DataFrame(rows)


# ── Legacy comparison (§10) ──────────────────────────────────────
def _parse_models(s) -> list[str]:
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return []
    items = {x.strip() for x in str(s).split(",") if x.strip()}
    return [m for m in MODEL_ORDER if m in items] + sorted(items - set(MODEL_ORDER))


def compare_with_legacy_mcs(corrected: pd.DataFrame, legacy_path: Path | None) -> pd.DataFrame:
    """Before/after membership table. The legacy file is only ever read."""
    base = corrected[["period", "option", "kappa", "n_common", "included_models", "mcs_size"]].rename(
        columns={"included_models": "corrected_mcs_R", "mcs_size": "corrected_mcs_size"}).copy()
    base["kappa"] = base["kappa"].astype(float).round(2)
    empty = dict(legacy_mcs="", legacy_mcs_size=np.nan, membership_changed=np.nan,
                 added_models="", removed_models="")
    if legacy_path is None or not Path(legacy_path).is_file():
        return base.assign(**empty, comparison_status="legacy_unavailable")
    legacy = pd.read_csv(legacy_path)
    need = {"period", "option", "kappa", "mcs"}
    if not need.issubset(legacy.columns):
        return base.assign(**empty, comparison_status=f"legacy_unparseable: needs {sorted(need)}")
    legacy = legacy[["period", "option", "kappa", "mcs"]].copy()
    legacy["kappa"] = pd.to_numeric(legacy["kappa"]).astype(float).round(2)
    merged = base.merge(legacy, on=["period", "option", "kappa"], how="left")
    out_rows = []
    for _, r in merged.iterrows():
        new = _parse_models(r["corrected_mcs_R"])
        if pd.isna(r["mcs"]):
            out_rows.append({**r.drop(labels=["mcs"]).to_dict(), **empty,
                             "comparison_status": "cell_missing_in_legacy"})
            continue
        old = _parse_models(r["mcs"])
        out_rows.append({
            **r.drop(labels=["mcs"]).to_dict(),
            "legacy_mcs": ",".join(old), "legacy_mcs_size": len(old),
            "membership_changed": set(old) != set(new),
            "added_models": ",".join(m for m in new if m not in old),
            "removed_models": ",".join(m for m in old if m not in new),
            "comparison_status": "compared",
        })
    return pd.DataFrame(out_rows)


def method_disagreements(primary: pd.DataFrame, sensitivity: pd.DataFrame) -> pd.DataFrame:
    key = ["period", "option", "kappa"]
    m = primary[key + ["n_common", "included_models"]].merge(
        sensitivity[key + ["included_models"]], on=key, suffixes=("_R", "_max"))
    return m[m["included_models_R"] != m["included_models_max"]].reset_index(drop=True)


# ── LaTeX (§10) ──────────────────────────────────────────────────
_TEX_MODEL = {"GBM": r"\GBM", "Heston": r"\Heston", "SBTS": r"\SBTS"}
_TEX_PERIOD = {"COVID_2019_2020": "COVID 2019--2020",
               "PostCOVID_2021_22": "PostCOVID 2021--2022",
               "Recent_2023_25": "Recent 2023--2025"}
_TEX_OPTION = {"basket_asian_call": "Basket call", "asian_worst_of_put": "Worst-of put"}


def render_latex_table(primary: pd.DataFrame, sensitivity: pd.DataFrame, reps: int = REPS) -> str:
    key = ["period", "option", "kappa"]
    t = primary.merge(sensitivity[key + ["included_models"]], on=key, suffixes=("", "_max"))

    def fmt(s):
        return r"\{" + ", ".join(_TEX_MODEL.get(m, m) for m in _parse_models(s)) + r"\}"

    lines = [
        r"\begin{table}[H]", r"\centering", r"\small",
        r"\caption{Model confidence set (Hansen--Lunde--Nason), recalculated on "
        r"common \emph{actual} seeds. Loss: Phase~2 $\sigma(R)$; $\alpha = 0.10$; "
        r"circular block bootstrap with block size~1 and "
        + f"{reps:,}".replace(",", "{,}") +
        r" replications. Primary statistic $T_R$; sensitivity $T_{\max}$. "
        r"$n$ is the number of common actual seeds. "
        r"$^{\dagger}$Worst-of put, $\kappa = 0.95$: \SBTS\ canonical seed~3 is the "
        r"converged retry with actual seed~10, so actual seed~3 (\GBM, \Heston) and "
        r"actual seed~10 (\SBTS) are not paired and these three cells use $n = 9$; "
        r"all other cells use $n = 10$.}",
        r"\label{tab:mcs-corrected}",
        r"\begin{tabular}{@{}lllcll@{}}", r"\toprule",
        r"Period & Option & $\kappa$ & $n$ & MCS ($T_R$) & MCS ($T_{\max}$) \\",
        r"\midrule",
    ]
    for i, period in enumerate(PERIODS):
        sub = t[t["period"] == period]
        first = True
        for option in OPTIONS:
            for kappa in KAPPAS:
                r = sub[(sub["option"] == option) & _kappa_mask(sub["kappa"], kappa)].iloc[0]
                n = f"{int(r['n_common'])}" + (r"$^{\dagger}$" if bool(r["affected_cell"]) else "")
                plabel = _TEX_PERIOD[period] if first else ""
                first = False
                lines.append(f"  {plabel} & {_TEX_OPTION[option]} & {kappa:.2f} & {n} & "
                             f"{fmt(r['included_models'])} & {fmt(r['included_models_max'])} \\\\")
        if i < len(PERIODS) - 1:
            lines.append(r"\midrule")
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}", ""]
    return "\n".join(lines)


# ── Synthetic smoke tests (§8 cell 7) ────────────────────────────
def _synthetic_cell(means: dict, noise: float, rng_seed: int,
                    period="COVID_2019_2020", option="asian_worst_of_put", kappa=0.95,
                    seeds=CANONICAL_SEEDS) -> pd.DataFrame:
    rng = np.random.default_rng(rng_seed)
    rows = []
    for ds in MODEL_ORDER:
        for s in seeds:
            v = means[ds] + noise * rng.standard_normal()
            rows.append({"ds": ds, "option": option, "kappa": kappa, "seed": int(s),
                         "phase": PHASE_RUN, "period": period, "std": v,
                         "cvar95": v, "mean": 0.0, "V0": 0.0})
    return pd.DataFrame(rows)


def run_synthetic_smoke_tests(reps: int = REPS) -> pd.DataFrame:
    results = []

    def record(name, fn):
        try:
            ok, detail = fn()
        except Exception as e:  # a smoke test must never crash the notebook silently
            ok, detail = False, f"{type(e).__name__}: {e}"
        results.append({"test": name, "passed": bool(ok), "detail": str(detail)})

    cell = ("COVID_2019_2020", "asian_worst_of_put", 0.95)

    def clear_winner():
        df = attach_actual_seed(_synthetic_cell({"GBM": .05, "Heston": .05, "SBTS": .01}, .001, 1), {})
        L, _ = build_mcs_loss_matrix(df, *cell)
        out = {m: run_arch_mcs(L, m, ALPHA, reps, 11)["included_models"] for m in ("R", "max")}
        return all(v == ["SBTS"] for v in out.values()), out

    def indistinguishable():
        df = attach_actual_seed(_synthetic_cell({"GBM": .05, "Heston": .05, "SBTS": .05}, .005, 2), {})
        L, _ = build_mcs_loss_matrix(df, *cell)
        out = {m: run_arch_mcs(L, m, ALPHA, reps, 12)["included_models"] for m in ("R", "max")}
        return all(v == list(MODEL_ORDER) for v in out.values()), out

    def replaced_seed_alignment():
        raw = _synthetic_cell({"GBM": .05, "Heston": .04, "SBTS": .03}, .002, 3)
        df = attach_actual_seed(raw, {("SBTS", "asian_worst_of_put", 0.95, 3): 10})
        L, a = build_mcs_loss_matrix(df, *cell)
        verify_matrix_pairing(df, L, *cell)
        pm = a["per_model"]
        ok = (a["common_actual_seeds"] == list(AFFECTED_COMMON_SEEDS)
              and pm["GBM"]["excluded_actual_seeds"] == [3]
              and pm["Heston"]["excluded_actual_seeds"] == [3]
              and pm["SBTS"]["excluded_actual_seeds"] == [10]
              and pm["SBTS"]["replacement_count"] == 1
              and 3 not in L.index and 10 not in L.index
              and list(L.columns) == list(MODEL_ORDER))
        return ok, f"n_common={a['n_common']}, common={a['common_actual_seeds']}"

    def missing_seed():
        raw = _synthetic_cell({"GBM": .05, "Heston": .04, "SBTS": .03}, .002, 4)
        raw = raw[~((raw["ds"] == "Heston") & (raw["seed"] == 7))]
        df = attach_actual_seed(raw, {})
        L, a = build_mcs_loss_matrix(df, *cell)
        ok = 7 not in L.index and a["n_common"] == 9 and a["per_model"]["GBM"]["excluded_actual_seeds"] == [7]
        return ok, f"n_common={a['n_common']}"

    def duplicate_rejected():
        raw = _synthetic_cell({"GBM": .05, "Heston": .04, "SBTS": .03}, .002, 5)
        df = attach_actual_seed(pd.concat([raw, raw.iloc[[0]]], ignore_index=True), {})
        try:
            build_mcs_loss_matrix(df, *cell)
        except ValueError as e:
            return "Duplicate" in str(e), str(e)
        return False, "duplicate not rejected"

    def nonfinite_rejected():
        raw = _synthetic_cell({"GBM": .05, "Heston": .04, "SBTS": .03}, .002, 6)
        raw.loc[raw.index[5], "std"] = np.inf
        try:
            build_mcs_loss_matrix(attach_actual_seed(raw, {}), *cell)
        except ValueError as e:
            return "Non-finite" in str(e), str(e)
        return False, "non-finite not rejected"

    def file_seed_mismatch_rejected():
        raw = _synthetic_cell({"GBM": .05, "Heston": .04, "SBTS": .03}, .002, 7)
        raw["file_seed"] = raw["seed"]  # claims SBTS seed 3 is actual 3: contradicts mapping
        try:
            attach_actual_seed(raw, {("SBTS", "asian_worst_of_put", 0.95, 3): 10})
        except ValueError as e:
            return "file_seed" in str(e), "rejected"
        return False, "mismatch not rejected"

    def missing_csv_fails():
        try:
            load_legacy_metrics(Path("/nonexistent/per_seed_metrics_3p.csv"))
        except FileNotFoundError:
            return True, "FileNotFoundError"
        return False, "no error"

    def rng_seed_stable():
        s1 = cell_rng_seed("COVID_2019_2020", "asian_worst_of_put", 0.95, "R")
        s2 = cell_rng_seed("COVID_2019_2020", "asian_worst_of_put", 0.95, "R")
        s3 = cell_rng_seed("COVID_2019_2020", "asian_worst_of_put", 0.95, "max")
        return s1 == s2 and s1 != s3 and 0 <= s1 < 2**32, f"R={s1}, max={s3}"

    def mcs_deterministic():
        df = attach_actual_seed(_synthetic_cell({"GBM": .05, "Heston": .049, "SBTS": .048}, .002, 8), {})
        L, _ = build_mcs_loss_matrix(df, *cell)
        a, b = run_arch_mcs(L, "R", ALPHA, reps, 99), run_arch_mcs(L, "R", ALPHA, reps, 99)
        return a == b, a["pvalues"]

    for name, fn in [
        ("clear winner -> {SBTS}", clear_winner),
        ("indistinguishable -> all three", indistinguishable),
        ("replaced seed 3->10 excluded from pairing (n=9)", replaced_seed_alignment),
        ("missing seed dropped from intersection", missing_seed),
        ("duplicate actual seed rejected", duplicate_rejected),
        ("non-finite loss rejected", nonfinite_rejected),
        ("file_seed contradicting mapping rejected", file_seed_mismatch_rejected),
        ("absent legacy CSV fails loudly", missing_csv_fails),
        ("cell RNG seed stable", rng_seed_stable),
        ("same seed -> identical MCS", mcs_deterministic),
    ]:
        record(name, fn)
    return pd.DataFrame(results)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 5 — Input provenance (fails loudly if the legacy CSV is absent)
# ═══════════════════════════════════════════════════════════════════
if OUTPUT_DIR.resolve() == LEGACY_RESULTS_DIR.resolve():
    raise RuntimeError("OUTPUT_DIR must not be the legacy results directory")

# Byte-level snapshot of every legacy artifact (excluding our own output dir),
# re-checked in the final gate to prove nothing legacy was modified.
LEGACY_SNAPSHOT_BEFORE = snapshot_tree(LEGACY_RESULTS_DIR, exclude=OUTPUT_DIR)
LEGACY_MCS_PRESENT = LEGACY_MCS_CSV.is_file()
LEGACY_MCS_SHA_BEFORE = sha256_file(LEGACY_MCS_CSV) if LEGACY_MCS_PRESENT else None

raw_df, INPUT_INFO = load_legacy_metrics(INPUT_CSV)
INPUT_SHA256 = INPUT_INFO["sha256"]

print(f"Resolved path : {INPUT_INFO['path']}")
print(f"SHA-256       : {INPUT_SHA256}")
print(f"Rows x cols   : {INPUT_INFO['n_rows']} x {INPUT_INFO['n_cols']}")
print(f"Columns       : {INPUT_INFO['columns']}")
print(f"file_seed col : {'present' if 'file_seed' in raw_df.columns else 'absent (derived from mapping)'}")
print(f"\nLegacy MCS CSV: {'present, sha256=' + LEGACY_MCS_SHA_BEFORE if LEGACY_MCS_PRESENT else 'absent'}")
print(f"Legacy files snapshotted: {len(LEGACY_SNAPSHOT_BEFORE)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 6 — Seed provenance: canonical_seed / actual_seed / replacement_used
# ═══════════════════════════════════════════════════════════════════
df = attach_actual_seed(raw_df, SEED_REPLACEMENTS)

print("Approved replacements:")
for (ds, opt, k, cs), a in SEED_REPLACEMENTS.items():
    print(f"  {ds} / {opt} / kappa={k:.2f} / canonical seed {cs} -> actual seed {a}")

replaced = df[df["replacement_used"]]
print(f"\nRows with replacement_used=True: {len(replaced)}")
print(replaced[["ds", "option", "kappa", "phase", "period", "canonical_seed",
                "actual_seed", "std"]].sort_values(["phase", "period"]).to_string(index=False))

print("\nreplacement_used counts by (ds, option, kappa):")
print(df.groupby(["ds", "option", "kappa"])["replacement_used"].sum()
        .loc[lambda s: s > 0].to_string())

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 7 — Hard validation (schema, finiteness, duplicates, coverage, seeds)
# ═══════════════════════════════════════════════════════════════════
VALIDATION = validate_legacy_metrics(df, SEED_REPLACEMENTS)
print(VALIDATION.to_string(index=False))
if not VALIDATION["passed"].all():
    raise RuntimeError("Hard validation failed — MCS not run. See table above.")
print("\n✓ Hard validation passed")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 8 — Loss-matrix builder: all 18 cells + expected-alignment gate
# ═══════════════════════════════════════════════════════════════════
MATRICES = {}
for period in PERIODS:
    for option in OPTIONS:
        for kappa in KAPPAS:
            losses, audit = build_mcs_loss_matrix(df, period, option, kappa)
            verify_matrix_pairing(df, losses, period, option, kappa)
            MATRICES[(period, option, kappa)] = (losses, audit)

ALIGNMENT_AUDIT = alignment_audit_table(MATRICES)
compact = (ALIGNMENT_AUDIT.groupby(["period", "option", "kappa"])
           .agg(n_common=("n_common", "first"),
                common_actual_seeds=("common_actual_seeds", "first"),
                expected=("expected_common_actual_seeds", "first"),
                status=("alignment_status", "first"))
           .reset_index())
if (ALIGNMENT_AUDIT["alignment_status"] != "ok").any():
    print(compact.to_string(index=False))
    raise RuntimeError("Observed alignment differs from the design expectation — MCS not run.")

print(compact.to_string(index=False))
print(f"\nn_common distribution: {compact['n_common'].value_counts().sort_index().to_dict()}")

# Display one affected cell
AFFECTED_SHOW = AFFECTED_CELLS[0]
L_show, A_show = MATRICES[AFFECTED_SHOW]
print(f"\nAffected cell {AFFECTED_SHOW} — loss matrix (index = actual_seed):")
print(L_show.to_string())
for ds, pm in A_show["per_model"].items():
    print(f"  {ds:<6s} actual={pm['actual_seeds']}  excluded={pm['excluded_actual_seeds']}  "
          f"{pm['exclusion_reason']}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 9 — Synthetic smoke tests (frozen MCS settings, synthetic data)
# ═══════════════════════════════════════════════════════════════════
SMOKE = run_synthetic_smoke_tests()
print(SMOKE.to_string(index=False))
if not SMOKE["passed"].all():
    raise RuntimeError("Synthetic smoke tests failed — do not trust real-data MCS.")
print("\n✓ All synthetic smoke tests passed")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 10 — Primary MCS: method R, all 18 cells
# ═══════════════════════════════════════════════════════════════════
import time

t0 = time.time()
PRIMARY, PVALS_R = run_mcs_all_cells(df, PRIMARY_METHOD, input_sha256=INPUT_SHA256,
                                     arch_version=VERSIONS["arch"])
print(f"Method {PRIMARY_METHOD}: {len(PRIMARY)} cells in {time.time() - t0:.1f}s\n")
print(PRIMARY[["period", "option", "kappa", "n_common", "included_models", "excluded_models",
               "gbm_pvalue", "heston_pvalue", "sbts_pvalue",
               "gbm_mean_common", "heston_mean_common", "sbts_mean_common"]].to_string(index=False))

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 11 — Sensitivity MCS: method max, otherwise identical settings
# ═══════════════════════════════════════════════════════════════════
t0 = time.time()
SENSITIVITY, PVALS_MAX = run_mcs_all_cells(df, SENSITIVITY_METHOD, input_sha256=INPUT_SHA256,
                                           arch_version=VERSIONS["arch"])
print(f"Method {SENSITIVITY_METHOD}: {len(SENSITIVITY)} cells in {time.time() - t0:.1f}s\n")
print(SENSITIVITY[["period", "option", "kappa", "n_common", "included_models",
                   "excluded_models"]].to_string(index=False))

DISAGREE = method_disagreements(PRIMARY, SENSITIVITY)
print(f"\nCells where method R and method max disagree: {len(DISAGREE)}")
if len(DISAGREE):
    print(DISAGREE.to_string(index=False))

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 12 — Legacy comparison (read-only; never an input to MCS)
# ═══════════════════════════════════════════════════════════════════
COMPARISON = compare_with_legacy_mcs(PRIMARY, LEGACY_MCS_CSV if LEGACY_MCS_PRESENT else None)
status = COMPARISON["comparison_status"].iloc[0]
print(f"Comparison status: {status}")
if (COMPARISON["comparison_status"] == "compared").any():
    changed = COMPARISON[COMPARISON["membership_changed"] == True]  # noqa: E712
    print(f"Cells with changed membership: {len(changed)} / {len(COMPARISON)}")
    print(COMPARISON[["period", "option", "kappa", "n_common", "legacy_mcs",
                      "corrected_mcs_R", "membership_changed"]].to_string(index=False))
    print("\nChanged cells are results to report, not implementation failures.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 13 — Output writer (CSV + LaTeX). Metadata JSON and the Markdown
# report are written by CELL 14 because they carry the final gate status.
# ═══════════════════════════════════════════════════════════════════
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WRITTEN = []


def write_output(name, content):
    path = OUTPUT_DIR / name
    if path.resolve().parent != OUTPUT_DIR.resolve():
        raise RuntimeError(f"Refusing to write outside OUTPUT_DIR: {path}")
    if isinstance(content, pd.DataFrame):
        content.to_csv(path, index=False, lineterminator="\n")
    else:
        path.write_text(content, encoding="utf-8")
    WRITTEN.append(path)
    print(f"  💾 {path}")


PVALUES_LONG = pd.concat([PVALS_R, PVALS_MAX], ignore_index=True)

write_output("mcs_corrected_method_R.csv", PRIMARY)
write_output("mcs_sensitivity_method_max.csv", SENSITIVITY)
write_output("mcs_pvalues_long.csv", PVALUES_LONG)
write_output("mcs_alignment_audit.csv", ALIGNMENT_AUDIT)
write_output("mcs_comparison_with_legacy.csv", COMPARISON)
write_output("tab_mcs_corrected.tex", render_latex_table(PRIMARY, SENSITIVITY))

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 14 — Final gate: acceptance criteria (§11), metadata JSON, report
# ═══════════════════════════════════════════════════════════════════
# Determinism: re-run both methods from scratch and compare CSV bytes.
rerun_R, rerun_pR = run_mcs_all_cells(df, PRIMARY_METHOD, input_sha256=INPUT_SHA256,
                                      arch_version=VERSIONS["arch"])
rerun_M, rerun_pM = run_mcs_all_cells(df, SENSITIVITY_METHOD, input_sha256=INPUT_SHA256,
                                      arch_version=VERSIONS["arch"])


def _csv(d):
    return d.to_csv(index=False, lineterminator="\n")


deterministic = (_csv(rerun_R) == (OUTPUT_DIR / "mcs_corrected_method_R.csv").read_text()
                 and _csv(rerun_M) == (OUTPUT_DIR / "mcs_sensitivity_method_max.csv").read_text()
                 and _csv(pd.concat([rerun_pR, rerun_pM], ignore_index=True))
                 == (OUTPUT_DIR / "mcs_pvalues_long.csv").read_text())

legacy_after = snapshot_tree(LEGACY_RESULTS_DIR, exclude=OUTPUT_DIR)
legacy_mcs_sha_after = sha256_file(LEGACY_MCS_CSV) if LEGACY_MCS_CSV.is_file() else None
aff = PRIMARY[PRIMARY["affected_cell"]]
all_losses = [L for L, _ in MATRICES.values()]

no_3_10_pairing = all(
    3 not in MATRICES[c][0].index and 10 not in MATRICES[c][0].index for c in AFFECTED_CELLS
) and all(  # every entry re-derived from the source row with the same actual seed
    verify_matrix_pairing(df, L, *c) for c, (L, _) in MATRICES.items())

GATE = [
    ("Input SHA-256 and resolved path recorded",
     bool(INPUT_SHA256) and Path(INPUT_INFO["path"]).is_absolute()),
    ("Exactly 18 primary MCS cells", len(PRIMARY) == 18),
    ("Exactly 3 cells with n_common=9", int((PRIMARY["n_common"] == 9).sum()) == 3),
    ("Exactly 15 cells with n_common=10", int((PRIMARY["n_common"] == 10).sum()) == 15),
    ("Affected common seed set is 0,1,2,4,5,6,7,8,9",
     len(aff) == 3 and set(aff["common_actual_seeds"]) == {"0,1,2,4,5,6,7,8,9"}
     and (aff["n_common"] == 9).all()),
    ("No row pairs GBM/Heston actual seed 3 with SBTS actual seed 10", no_3_10_pairing),
    ("No duplicate (cell, ds, actual_seed)",
     not df[df["phase"] == PHASE_RUN].duplicated(
         ["period", "option", "kappa", "ds", "actual_seed"]).any()),
    ("All losses supplied to MCS are finite",
     all(np.isfinite(L.to_numpy(dtype=float)).all() for L in all_losses)),
    ("Model order identical in all cells",
     all(list(L.columns) == list(MODEL_ORDER) for L in all_losses)),
    ("Re-run in same environment yields identical CSV content", deterministic),
    ("Synthetic clear-winner and tie smoke tests pass", bool(SMOKE["passed"].all())),
    ("Primary and sensitivity outputs saved",
     (OUTPUT_DIR / "mcs_corrected_method_R.csv").is_file()
     and (OUTPUT_DIR / "mcs_sensitivity_method_max.csv").is_file()),
    ("Old MCS artifact never overwritten", legacy_mcs_sha_after == LEGACY_MCS_SHA_BEFORE),
    ("No checkpoint/notebook written; all writes inside OUTPUT_DIR",
     all(p.resolve().parent == OUTPUT_DIR.resolve() and p.suffix not in {".pt", ".pth", ".ipynb"}
         for p in WRITTEN)),
    ("Legacy descriptive artifacts byte-for-byte untouched", legacy_after == LEGACY_SNAPSHOT_BEFORE),
]
GATE_DF = pd.DataFrame(GATE, columns=["criterion", "passed"])
GATE_STATUS = "PASS" if GATE_DF["passed"].all() else "FAIL"

# ── Metadata JSON ────────────────────────────────────────────────
rng_seeds = [{"period": r.period, "option": r.option, "kappa": r.kappa,
              "method": r.method, "rng_seed": int(r.rng_seed)}
             for d in (PRIMARY, SENSITIVITY) for r in d.itertuples()]
metadata = {
    "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "design_document": "MCS_LEGACY_RESULTS_RECALCULATION_DESIGN.md",
    "source": {k: INPUT_INFO[k] for k in ("path", "sha256", "n_rows", "n_cols")},
    "legacy_mcs": {"path": str(LEGACY_MCS_CSV), "present": LEGACY_MCS_PRESENT,
                   "sha256": LEGACY_MCS_SHA_BEFORE},
    "git_commit": GIT_COMMIT,
    "versions": VERSIONS,
    "seed_replacements": [
        {"ds": k[0], "option": k[1], "kappa": k[2], "canonical_seed": k[3], "actual_seed": v}
        for k, v in SEED_REPLACEMENTS.items()],
    "mcs_parameters": {
        "implementation": "arch.bootstrap.MCS", "phase": PHASE_RUN, "loss": LOSS_COL,
        "models": list(MODEL_ORDER), "alpha": ALPHA, "reps": REPS, "block_size": BLOCK_SIZE,
        "bootstrap": BOOTSTRAP, "primary_method": PRIMARY_METHOD,
        "sensitivity_method": SENSITIVITY_METHOD,
        "rng_seed_rule": f"sha256('{RNG_KEY_VERSION}|period|option|kappa:.2f|method')[:8] mod 2**32",
        "alignment": "intersection of actual seeds per cell",
    },
    "rng_seeds": rng_seeds,
    "total_cells": int(len(PRIMARY)),
    "n_common_distribution": {str(k): int(v) for k, v in
                              PRIMARY["n_common"].value_counts().sort_index().items()},
    "affected_cells": [{"period": p, "option": o, "kappa": k} for p, o, k in AFFECTED_CELLS],
    "method_disagreements": int(len(DISAGREE)),
    "legacy_comparison_status": str(COMPARISON["comparison_status"].iloc[0]),
    "final_gate": {"status": GATE_STATUS,
                   "checks": [{"criterion": c, "passed": bool(p)} for c, p in GATE]},
    "paired_t_tests_recomputed": False,
    "descriptive_results_changed": False,
    "outputs": list(OUTPUT_FILES),
}

# ── Markdown report ──────────────────────────────────────────────
def _md(d):
    cols = list(d.columns)
    out = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    out += ["| " + " | ".join(str(v) for v in row) + " |" for row in d.itertuples(index=False)]
    return "\n".join(out)


main_tbl = PRIMARY.merge(SENSITIVITY[["period", "option", "kappa", "included_models"]],
                         on=["period", "option", "kappa"], suffixes=("_R", "_max"))
main_tbl = main_tbl[["period", "option", "kappa", "n_common", "included_models_R",
                     "included_models_max", "gbm_mean_common", "heston_mean_common",
                     "sbts_mean_common"]].copy()
for c in ("gbm_mean_common", "heston_mean_common", "sbts_mean_common"):
    main_tbl[c] = main_tbl[c].map(lambda v: f"{v:.4f}")

report = [
    "# MCS recalculation report (legacy thesis results)", "",
    f"* Generated (UTC): {metadata['timestamp_utc']}",
    f"* Input: `{INPUT_INFO['path']}`",
    f"* Input SHA-256: `{INPUT_SHA256}`",
    f"* Versions: {VERSIONS}; git commit: {GIT_COMMIT}",
    f"* **Final gate: {GATE_STATUS}**", "",
    "## 1. Three separate concepts", "",
    "1. **Legacy descriptive estimates — unchanged.** They retain all ten available replicates, "
    "including SBTS actual seed 10 standing in for canonical seed 3 "
    "(`asian_worst_of_put`, κ=0.95). The `*_mean_all_available` columns reproduce them.",
    "2. **Corrected MCS estimand.** Loss matrix rows are common *actual* seeds: n = 9 in the "
    "three affected cells (seeds 0,1,2,4,5,6,7,8,9) and n = 10 in the other 15 cells. "
    "Primary method `R`.",
    "3. **Sensitivity result.** Method `max`, otherwise identical settings. Disagreements with "
    "`R` are listed below, not hidden.", "",
    "## 2. Settings", "",
    f"`arch.bootstrap.MCS` {VERSIONS['arch']}; phase `{PHASE_RUN}`; loss `{LOSS_COL}`; "
    f"α = {ALPHA}; {REPS:,} reps; `{BOOTSTRAP}` bootstrap; block size {BLOCK_SIZE}; "
    f"model order {list(MODEL_ORDER)}; cell RNG seeds from SHA-256 (see `mcs_metadata.json`).", "",
    "## 3. Results (common actual seeds)", "",
    _md(main_tbl), "",
    "## 4. Sensitivity: method R vs method max", "",
    (f"{len(DISAGREE)} cell(s) disagree:\n\n" + _md(DISAGREE)) if len(DISAGREE)
    else "Methods `R` and `max` give identical membership in all 18 cells.", "",
    "## 5. Before/after comparison with the legacy MCS", "",
]
if (COMPARISON["comparison_status"] == "compared").any():
    n_changed = int((COMPARISON["membership_changed"] == True).sum())  # noqa: E712
    report += [
        f"Legacy file `{LEGACY_MCS_CSV}` (sha256 `{LEGACY_MCS_SHA_BEFORE}`) was read for "
        f"comparison only. Membership changed in {n_changed} of {len(COMPARISON)} cells. "
        "Changed cells are results to report; they were not forced to match the prior narrative.", "",
        _md(COMPARISON[["period", "option", "kappa", "n_common", "legacy_mcs",
                        "corrected_mcs_R", "membership_changed", "added_models",
                        "removed_models"]]), ""]
else:
    report += [f"No comparison: {COMPARISON['comparison_status'].iloc[0]} (`{LEGACY_MCS_CSV}`).", ""]
report += [
    "## 6. Scientific boundary", "",
    "* This run recalculates the MCS only. `descriptive_results_changed = false`.",
    "* The 216 paired t-tests were **not** recomputed (`paired_t_tests_recomputed = false`). "
    "This MCS correction does not certify or repair any separate paired-test alignment issue.",
    "* No thesis conclusion is changed automatically.", "",
    "## 7. Acceptance gate", "",
    _md(GATE_DF.assign(passed=GATE_DF["passed"].map({True: "PASS", False: "FAIL"}))), "",
]

write_output("mcs_metadata.json", json.dumps(metadata, indent=2, default=str) + "\n")
write_output("MCS_RECALCULATION_REPORT.md", "\n".join(report))

missing_outputs = [f for f in OUTPUT_FILES if not (OUTPUT_DIR / f).is_file()]
if missing_outputs:
    GATE_STATUS = "FAIL"
    print(f"Missing outputs: {missing_outputs}")

print("\n" + GATE_DF.assign(passed=GATE_DF["passed"].map({True: "PASS", False: "FAIL"}))
      .to_string(index=False))
print("\n" + "═" * 60)
print(f" FINAL GATE: {GATE_STATUS} ".center(60, "═"))
print("═" * 60)
if GATE_STATUS != "PASS":
    raise RuntimeError("Final gate FAILED — do not use these outputs in the thesis.")